# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to programmatically load and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` fields.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}; Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We use the `@id` field to reference all entities, record sets, and fields.

In [ ]:
# List all available record sets and their fields by `@id`
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    print("  Fields:")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            fid = field.get('@id', str(field))
        else:
            fid = field
        print(f"    - Field @id: {fid}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

In the FAIR^2 dataset, the protein quantitation data is in the main record set (for this dataset, the first record set). We reference by its `@id`. All field columns also use their `@id`.

In [ ]:
# Extract data from all record sets by their @id
# For demonstration, we load all; you may choose those relevant to your analysis
dataframes = dict()
for rs in record_sets:
    rs_id = rs["@id"]
    print(f"Loading records from RecordSet {rs_id} ...")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        print(f"  -> Loaded {len(df)} records, columns: {df.columns.tolist()}")
        dataframes[rs_id] = df
    except Exception as e:
        print(f"  -> Could not load records: {e}")

# Example: print head of the main protein data record set
main_rs_id = record_sets[0]['@id']
print(f"Columns in main RecordSet ({main_rs_id}):")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping, referencing the relevant field `@id`s.

For demonstration:
- We filter proteins with a high percentage sequence coverage
- Normalize the `coverage_percent` column
- Group by a categorical field if present (e.g., by sample label if available)

**Note:** Replace these with domain-specific steps as needed. The exact field `@id`s and names can be verified in the Data Overview.

In [ ]:
# Replace this with the actual @id for coverage_percent from the field overview
numeric_field = None
for col in dataframes[main_rs_id].columns:
    if "coverage" in col.lower():  # heuristic: find coverage percentage field
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = dataframes[main_rs_id].columns[2]  # fallback to third column
print(f"Numeric field for coverage: {numeric_field}")

# Filter by coverage percent > 50
threshold = 50
filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field] > threshold]
print(f"Filtered proteins with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"First normalized values for {numeric_field}:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt grouping by a categorical field (e.g., `sample_label`)
# Find a categorical column
categorical_field = None
for col in filtered_df.columns:
    if (filtered_df[col].dtype == object or filtered_df[col].dtype.name == "category") and not col.endswith("normalized"):
        categorical_field = col
        break
if categorical_field is not None:
    grouped_df = filtered_df.groupby(categorical_field).mean(numeric_only=True)
    print(f"Grouped normalized {numeric_field} by {categorical_field}:")
    print(grouped_df[[f"{numeric_field}_normalized"]].head())

## 5. Visualization
Visualize the distribution of `coverage_percent` and any relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of coverage_percent
plt.figure(figsize=(8, 4))
sns.histplot(dataframes[main_rs_id][numeric_field], bins=40, color="skyblue")
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Protein count")
plt.show()

# If there is a molecular weight column, plot coverage vs. MW
mw_field = None
for col in dataframes[main_rs_id].columns:
    if "weight" in col.lower() or "mw" in col.lower():
        mw_field = col
        break
if mw_field:
    plt.figure(figsize=(7,5))
    sns.scatterplot(
        x=dataframes[main_rs_id][mw_field],
        y=dataframes[main_rs_id][numeric_field],
        alpha=0.7
    )
    plt.xlabel(mw_field)
    plt.ylabel(numeric_field)
    plt.title(f"{numeric_field} vs. {mw_field}")
    plt.show()

## 6. Conclusion

- We loaded the FAIR^2 protein dataset using `mlcroissant` and explored its schema entities by their `@id`.
- We demonstrated programmatic, reproducible EDA using only schema-driven references to fields/columns.
- This notebook can be adapted for further domain-specific analysis using unique `@id` references as required by best FAIR practices.